# 🎬 NotyCaption Pro - Caption Generator

Optimized for **Google Colab T4 GPU**.


In [ ]:
# Check GPU
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
print('GPU Name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')


In [ ]:
# Install dependencies
!apt-get update -qq && apt-get install -y ffmpeg -qq
!pip install -q -U openai-whisper pysrt pysubs2 google-auth google-auth-oauthlib google-api-python-client torch torchvision torchaudio
print('✅ Installed successfully')


In [ ]:
# Google Drive Auth
from google.colab import auth
auth.authenticate_user()

from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import io, json, os

creds, _ = default()
drive_service = build('drive', 'v3', credentials=creds)
print('✅ Drive Ready')


In [ ]:
# Audio Configuration
AUDIO_ID = '{{AUDIO_ID}}'
LANGUAGE = '{{LANGUAGE}}'
WORDS_PER_LINE = {{WORDS_PER_LINE}}
OUTPUT_FORMAT = '{{OUTPUT_FORMAT}}'
OPERATION_ID = '{{OPERATION_ID}}'
AUDIO_NAME = '{{AUDIO_NAME}}'

print(f'📁 Audio ID: {AUDIO_ID}')
print(f'📄 Output: {AUDIO_NAME}.{OUTPUT_FORMAT}')


In [ ]:
# Download Audio
def download_file(file_id, destination):
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.FileIO(destination, 'wb')
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()
        if status:
            print(int(status.progress()*100), '%')

audio_path = f'/content/audio_{OPERATION_ID}.wav'
download_file(AUDIO_ID, audio_path)
print('✅ Audio Downloaded')


In [ ]:
# Whisper with GPU (T4)
import whisper, torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

model = whisper.load_model('large-v3', device=device)

if LANGUAGE == 'auto' or LANGUAGE == '':
    result = model.transcribe(audio_path, word_timestamps=True, fp16=True)
else:
    result = model.transcribe(audio_path, language=LANGUAGE, word_timestamps=True, fp16=True)

print('✅ Transcription Done')


In [ ]:
# Generate Subtitles
subtitles = []
idx = 1

for seg in result['segments']:
    words = seg.get('words', [])
    for i in range(0, len(words), WORDS_PER_LINE):
        chunk = words[i:i+WORDS_PER_LINE]
        if not chunk:
            continue
        text = ' '.join([w['word'].strip() for w in chunk])
        subtitles.append((idx, chunk[0]['start'], chunk[-1]['end'], text))
        idx += 1

print('Lines:', len(subtitles))


In [ ]:
# Save SRT File
output_name = f'{AUDIO_NAME}.{OUTPUT_FORMAT}'
output_path = f'/content/{output_name}'

import pysrt
from datetime import timedelta

srt = pysrt.SubRipFile()

for i, s, e, t in subtitles:
    item = pysrt.SubRipItem(
        index=i,
        text=t,
        start=pysrt.SubRipTime(seconds=int(s), milliseconds=int((s % 1) * 1000)),
        end=pysrt.SubRipTime(seconds=int(e), milliseconds=int((e % 1) * 1000))
    )
    srt.append(item)

srt.save(output_path)
print(f'✅ Saved: {output_name}')


In [ ]:
# Upload to Drive
file_metadata = {'name': output_name}
media = MediaFileUpload(output_path, resumable=True)
uploaded = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()

print(f'✅ Uploaded File ID: {uploaded["id"]}')

# Send result back
result = {
    'success': True,
    'file_id': uploaded['id'],
    'file_name': output_name,
    'operation_id': OPERATION_ID
}

from IPython.display import display, Javascript
display(Javascript(f"sessionStorage.setItem('colab_result_{OPERATION_ID}', '{json.dumps(result)}'); console.log('✅ Result saved');"))

print('🎉 All done! Close this tab.')
